# 01 · Generate and validate the synthetic bioprosthetic-AVR cohort

Thin wrapper around the `simulator/` package. Everything here is reproducible from the command line:

```bash
/data/abar/alexenv/bin/python simulator/cli.py calibrate      # ~10 min: fits noise, death and per-class onset to the published curves
/data/abar/alexenv/bin/python simulator/cli.py validate       # acceptance tests -> simulator/output/validation_report.md + figures
/data/abar/alexenv/bin/python simulator/cli.py generate       # writes data/synthetic/*.csv + data_dictionary.md
```

What the simulator is and why it is valid: `docs/04_synthetic_cohort_spec.md` (design), `simulator/README.md` (implementation, provenance, calibration results). Calibration targets are the curves reconstructed from Kermen 2022, NOTION 10-year and Wakami 2022 in `km_reconstruct/` (design choice D6).

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, Image, display

SIM = Path('simulator').resolve(); sys.path.insert(0, str(SIM)); sys.path.insert(0, str(SIM.parent / 'km_reconstruct'))
from params import SimParams, default_params, provenance_table
from physics import fit_physics_constants
from simulate import simulate_cohort

calibrated = SIM / 'output' / 'params_calibrated.json'
p = SimParams.from_json(calibrated) if calibrated.exists() else default_params()
if not p.physics.c_model:
    fit_physics_constants(p)
print('calibrated parameters loaded' if calibrated.exists() else 'WARNING: uncalibrated defaults (run cli.py calibrate)')

## 1. Calibration report (physics → noise → death → onset per valve class → cross-checks)

Produced by `cli.py calibrate`. Each row is a published number the simulator had to reproduce with the paper's own estimator and single-echo label rule; the NOTION SAVR row marked *PREDICTED* was not fitted (Perimount and Trifecta classes transported to age 79 by the age hazard ratio).

In [ ]:
rep = SIM / 'output' / 'calibration_report.md'
display(Markdown(rep.read_text() if rep.exists() else '*no calibration report yet*'))

## 2. Generate the cohort (D5: 10,000 valves, SAVR:TAVR 55:45, implants 2010–2020, censored Sept 2026)

In [ ]:
out = simulate_cohort(p, n=10000, seed=1)
pv, ev, events, lab, lt = (out[k] for k in ('patients_valves', 'echo_visits', 'events', 'labels', 'latent_truth'))
print({k: len(out[k]) for k in ('patients_valves', 'echo_visits', 'events', 'labels', 'latent_truth')})
display(pv.head(3).T)
display(ev.head(12))
display(events.event_type.value_counts().to_frame('n'))

## 3. What the label looks like: single-echo vs confirmed, and the oracle

Cumulative incidence of the primary endpoint (confirmed VARC-3 stage ≥2, death and reintervention competing) by approach, next to the single-echo label and the noise-free oracle. The gap between the single-echo and confirmed curves is the label flicker Velders 2024 describes.

In [ ]:
from survival import aj, step_eval
grid = np.arange(0, 12.01, 0.25)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, appr in zip(axes, ('SAVR', 'TAVR')):
    d = lab.merge(pv[['valve_id', 'approach']], on='valve_id'); d = d[d.approach == appr]
    for col, name, ls in (('varc3_single2', 'VARC-3 single echo', ':'), ('varc3_conf2', 'VARC-3 confirmed (primary)', '-'), ('oracle_varc3_single2', 'oracle (noise-free)', '--')):
        t = np.where(np.isfinite(d[col]), d[col], d.followup_end_years)
        st = np.where(np.isfinite(d[col]), 1, np.where(d.death == 1, 2, 0))
        c = aj(t.to_numpy(), st)
        if 1 in c:
            ax.plot(grid, 100 * step_eval(grid, c[1][0], c[1][1]), ls, label=name)
    ax.set_title(appr); ax.set_xlabel('years since implant'); ax.grid(alpha=.3); ax.legend()
axes[0].set_ylabel('cumulative incidence (%)'); plt.show()

## 4. Acceptance tests (docs/04 §5) and figures

`cli.py validate` re-simulates cohorts matched to each paper and compares the simulated observed curves with the reconstructed published curves, checks the Velders label-flicker statistics, the reference-echo tables, mortality, reintervention, and the emergent hazard ratios.

In [ ]:
vrep = SIM / 'output' / 'validation_report.md'
display(Markdown(vrep.read_text() if vrep.exists() else '*run cli.py validate first*'))
for f in ('curves_sim_vs_target.png', 'reference_mpg_vs_ase.png', 'label_flicker_heatmap.png'):
    fp = SIM / 'output' / 'figures' / f
    if fp.exists():
        display(Image(filename=str(fp)))

## 5. Parameter provenance

Every parameter is tagged with its source or flagged as an assumption (assumptions get a sensitivity run).

In [ ]:
display(Markdown(provenance_table()))